<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 6 · Jueves — Benchmarking Sistemático</h1>
<h3>LazyPredict + GridSearchCV + Tabla comparativa profesional</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final vas a poder:

1. Construir un **benchmark sistemático** que compare 5+ modelos a la vez.
2. Usar **`LazyPredict`** para auto-probar **docenas de modelos** en una línea.
3. Optimizar hiperparámetros con **`GridSearchCV`**.
4. Entender el rol de la **cross-validation** en evaluación honesta.
5. Resolver **3 ejercicios** integradores.

> Requisito: pipelines (lunes), imputer (martes), Random Forest (miércoles).

# 1. ¿Por qué hacer benchmark?

Cada modelo tiene **fortalezas y debilidades**:

- **Regresión Lineal:** rápida, interpretable, asume linealidad.
- **KNN:** flexible, sin entrenamiento, lento al predecir.
- **Árbol único:** interpretable, tiende a overfittear.
- **Random Forest:** robusto, lento de entrenar, menos interpretable.
- **Boosting (XGBoost, LGBM):** suele ganar en datos tabulares, requiere tuning.

**Ningún modelo gana siempre.** Por eso un data scientist:

1. Prueba varios modelos
2. Compara con las mismas métricas
3. Elige el mejor en base a evidencia (no intuición)

Eso es **benchmarking**.

# 2. Benchmark manual — el flujo profesional

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Dataset desafiante
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Diccionario de modelos a comparar (con sus pipelines)
modelos = {
    'Lineal':           make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge':            make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso':            make_pipeline(StandardScaler(), Lasso(alpha=0.1)),
    'KNN(k=10)':        make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10, n_jobs=-1)),
    'Árbol(d=10)':      DecisionTreeRegressor(max_depth=10, random_state=42),
    'RandomForest(200)': RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42),
    'GradientBoosting':  GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
}

resultados = []
for nombre, modelo in modelos.items():
    t_inicio = time.time()
    modelo.fit(X_train, y_train)
    t_fit = time.time() - t_inicio

    t_inicio = time.time()
    pred = modelo.predict(X_test)
    t_pred = time.time() - t_inicio

    resultados.append({
        'Modelo': nombre,
        'R²':         r2_score(y_test, pred),
        'MAE':        mean_absolute_error(y_test, pred),
        'RMSE':       np.sqrt(mean_squared_error(y_test, pred)),
        'Tiempo fit (s)':  round(t_fit, 2),
        'Tiempo pred (s)': round(t_pred, 3)
    })

tabla = pd.DataFrame(resultados).set_index('Modelo').sort_values('R²', ascending=False).round(4)
tabla

In [ ]:
# Visualización del benchmark
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# R² por modelo
colores = sns.color_palette('viridis', len(tabla))
tabla['R²'].plot(kind='barh', ax=axes[0], color=colores, edgecolor='white')
for i, v in enumerate(tabla['R²']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold', fontsize=9)
axes[0].set_xlabel('R²')
axes[0].set_title('Ranking de modelos por R²', fontweight='bold')

# Tiempo de entrenamiento
tabla['Tiempo fit (s)'].plot(kind='barh', ax=axes[1], color='#C0504D', edgecolor='white')
for i, v in enumerate(tabla['Tiempo fit (s)']):
    axes[1].text(v + 0.05, i, f'{v}s', va='center', fontweight='bold', fontsize=9)
axes[1].set_xlabel('Segundos')
axes[1].set_title('Tiempo de entrenamiento', fontweight='bold')

plt.tight_layout()
plt.show()

print('💡 No solo importa el R² — también el tiempo. En producción, un modelo')
print('   que tarda 100x más por 1% más de R² puede no valer la pena.')

# 3. LazyPredict — el atajo para benchmarks gigantes

**LazyPredict** es una librería que entrena **~30 modelos diferentes** en una línea y te entrega la tabla comparativa. Ideal para una primera exploración rápida.

### Instalación

```bash
pip install lazypredict
```

### Uso

```python
from lazypredict.Supervised import LazyRegressor

reg = LazyRegressor(verbose=0, ignore_warnings=True)
modelos_train, predicciones = reg.fit(X_train, X_test, y_train, y_test)
```

> ⚠️ LazyPredict es para **exploración inicial**. No es para producción — te dice cuáles modelos vale la pena tunear.

Vamos a ejecutarlo (si no lo tienes, descomenta la línea de pip install):

In [ ]:
# !pip install lazypredict --quiet

In [ ]:
# Usamos una muestra para que sea rápido (LazyPredict es lento en datasets grandes)
np.random.seed(42)
idx = np.random.choice(X_train.index, size=3000, replace=False)
X_train_mini = X_train.loc[idx]
y_train_mini = y_train.loc[idx]

try:
    from lazypredict.Supervised import LazyRegressor
    reg = LazyRegressor(verbose=0, ignore_warnings=True)
    modelos_lazy, predicciones = reg.fit(X_train_mini, X_test, y_train_mini, y_test)
    
    # Mostrar top 10 por R²
    print('🏆 Top 10 modelos según LazyPredict:')
    print(modelos_lazy.head(10))
except ImportError:
    print('LazyPredict no instalado — corre la celda anterior con "!pip install lazypredict"')

# 4. `GridSearchCV` — encontrar los mejores hiperparámetros automáticamente

Ayer (miércoles) hicimos tuning manual con dobles loops. **GridSearchCV** automatiza eso:

1. Le pasas un **diccionario de hiperparámetros** a probar.
2. Hace todas las combinaciones posibles.
3. Usa **cross-validation** (k-fold) para evaluar cada una de forma honesta.
4. Te devuelve la mejor combinación.

### ¿Qué es cross-validation?

En vez de evaluar una sola vez en test, divide el train en `k` partes (típicamente 5 o 10), entrena en `k-1` y evalúa en la restante. Repite `k` veces. **Promedia los resultados**.

Esto da una estimación **más estable** del rendimiento del modelo.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Definir el modelo base + grilla de hiperparámetros a probar
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [10, 15, None]
}

# GridSearch con 3-fold CV (3 splits dentro del train) — total: 3×3×3 = 27 entrenamientos
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,                  # 3-fold CV
    scoring='r2',          # métrica a optimizar
    n_jobs=-1,             # paralelización
    verbose=1
)

print('Entrenando 27 combinaciones (3×3 hiperparámetros × 3 CV folds)...')
grid.fit(X_train, y_train)

print(f'\n🏆 Mejores hiperparámetros: {grid.best_params_}')
print(f'   Mejor R² (CV):           {grid.best_score_:.4f}')
print(f'   R² en test:              {grid.score(X_test, y_test):.4f}')

In [ ]:
# Ver TODAS las combinaciones probadas (ordenadas por rendimiento)
resultados_grid = pd.DataFrame(grid.cv_results_)
resultados_grid = resultados_grid[[
    'param_n_estimators', 'param_max_depth',
    'mean_test_score', 'std_test_score',
    'mean_fit_time'
]].sort_values('mean_test_score', ascending=False).round(4)

print(resultados_grid.to_string(index=False))

---
# 🏋️ Ejercicios largos — flujo profesional completo

Estos 2 ejercicios siguen el **flujo real de un data scientist**, de principio a fin:

1. 🔍 **Explorar y limpiar** los datos
2. 🔧 **Pipeline completo** con `ColumnTransformer` (imputación + escalado + encoding)
3. ⚡ **LazyPredict** primero — para ver rápidamente qué familias de modelos prometen
4. 🎯 **GridSearchCV** después — para afinar el mejor modelo
5. 💾 **Guardar el mejor modelo** con `joblib` para producción

> No hay solución incluida — es su turno de aplicar todo lo de la semana. Trabajen en equipos.

**Recuerden instalar LazyPredict si no lo tienen:**
```python
!pip install lazypredict --quiet
```

## 🚗 Ejercicio 1 — Predecir el precio de autos (Cars93)

Trabajas para una automotriz. Te dan un dataset de 93 modelos de autos de 1993 con sus características y precios. Tu misión: predecir el **precio** de un auto a partir de sus especificaciones.

**Dataset:**
```python
url = 'https://raw.githubusercontent.com/selva86/datasets/master/Cars93_miss.csv'
df = pd.read_csv(url)
df.head()
```

> 💡 Ojo al apellido `_miss` del archivo: este dataset **tiene valores faltantes a propósito**. Van a necesitar imputarlos dentro del pipeline.

**Columnas relevantes:**
- **Categóricas:** `Manufacturer`, `Type`, `AirBags`, `DriveTrain`, `Man.trans.avail`, `Origin`
- **Numéricas:** `MPG.city`, `MPG.highway`, `EngineSize`, `Horsepower`, `RPM`, `Fuel.tank.capacity`, `Passengers`, `Length`, `Weight`, etc.
- **Target:** `Price`
- ⚠️ **Excluir** `Min.Price`, `Max.Price` y `Model`/`Make` (son fugas o identificadores).

### Tu misión (flujo completo)

**Parte A — Explorar y limpiar**
1. Cargar, revisar `.info()`, `.describe()`, nulos por columna.
2. Eliminar las columnas que no se deben usar (`Min.Price`, `Max.Price`, `Model`, `Make`).
3. Quitar filas donde el **target `Price`** sea nulo (no se imputa el target).

**Parte B — Pipeline completo con ColumnTransformer**
4. Construir un `ColumnTransformer` que:
   - Para numéricas: `SimpleImputer(median)` + `StandardScaler`
   - Para categóricas: `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown='ignore')`
5. Train/test split 80/20, `random_state=42`.

**Parte C — LazyPredict primero**
6. Aplicar `LazyRegressor` sobre los datos preprocesados para ver **qué modelos prometen**.
   - Pista: como LazyPredict no acepta el pipeline directo, aplica el `preprocesador.fit_transform()` a train y `transform()` a test, y pásale los arrays ya transformados.
7. Mirar el top 5 de modelos por R².

**Parte D — GridSearchCV sobre el mejor**
8. Tomar uno de los mejores modelos del paso anterior (ej. RandomForest o GradientBoosting).
9. Meterlo en un pipeline completo (`ColumnTransformer` + modelo).
10. Hacer `GridSearchCV` con `cv=5` sobre al menos 2 hiperparámetros.
11. Reportar los mejores hiperparámetros y el R² en test.

**Parte E — Guardar el mejor modelo**
12. Guardar el pipeline ganador (ya entrenado) con `joblib.dump()`.
13. Cargarlo de nuevo y predecir el precio de **un auto inventado** por ustedes (creen el DataFrame con todas las columnas).

In [ ]:
# Parte A — Explorar y limpiar 👇



In [ ]:
# Parte B + C — Pipeline con ColumnTransformer + LazyPredict 👇



In [ ]:
# Parte D + E — GridSearchCV + guardar el mejor modelo 👇



## 🎓 Ejercicio 2 — Predecir el puntaje de matemáticas (Students Performance)

Trabajas en el área de analítica educativa de un colegio. Te dan datos de 1,000 estudiantes con su contexto socioeconómico y sus puntajes. Tu misión: predecir el **puntaje de matemáticas** de un estudiante.

**Dataset:**
```python
url = 'https://raw.githubusercontent.com/rashida048/Datasets/master/StudentsPerformance.csv'
df = pd.read_csv(url)
df.head()
```

**Columnas:**
- **Categóricas (5):** `gender`, `race/ethnicity`, `parental level of education`, `lunch`, `test preparation course`
- **Numéricas:** `reading score`, `writing score`
- **Target:** `math score`

> 💡 Este dataset está **limpio** (sin nulos), pero tiene **muchas categóricas** — es el escenario ideal para que el `OneHotEncoder` dentro del `ColumnTransformer` brille.

### Tu misión (mismo flujo completo)

**Parte A — Explorar**
1. Cargar y revisar `.info()`, `.describe()`.
2. Hacer al menos **2 visualizaciones**: distribución de `math score` y un boxplot de `math score` por alguna categórica (ej. `test preparation course` o `lunch`).

**Parte B — Pipeline completo con ColumnTransformer**
3. Definir `X` (todas menos `math score`) y `y = math score`.
4. Construir un `ColumnTransformer`:
   - Numéricas (`reading score`, `writing score`) → `StandardScaler`
   - Categóricas → `OneHotEncoder(handle_unknown='ignore')`
5. Train/test split 80/20, `random_state=42`.

**Parte C — LazyPredict primero**
6. Preprocesar con el `ColumnTransformer` y correr `LazyRegressor`.
7. Ver el top de modelos. ¿Cuáles familias funcionan mejor en este problema?

**Parte D — GridSearchCV sobre el mejor**
8. Elegir uno de los mejores y armar un pipeline completo.
9. `GridSearchCV` con `cv=5` sobre al menos 2 hiperparámetros.
10. Reportar mejores hiperparámetros + R², MAE y RMSE en test.

**Parte E — Guardar y usar el modelo**
11. Guardar el pipeline ganador con `joblib.dump()`.
12. Cargarlo y predecir el `math score` de un estudiante inventado:
    ```python
    estudiante = pd.DataFrame([{
        'gender': 'female',
        'race/ethnicity': 'group B',
        'parental level of education': "bachelor's degree",
        'lunch': 'standard',
        'test preparation course': 'completed',
        'reading score': 72,
        'writing score': 74
    }])
    ```

**Pregunta de cierre para discutir:** ¿qué tan útil es predecir `math score` usando `reading score` y `writing score`? ¿No será que esas 3 notas están tan correlacionadas que el modelo es casi trivial? Comenten qué pasaría si **quitan** reading y writing y solo usan las categóricas.

In [ ]:
# Parte A — Explorar + visualizaciones 👇



In [ ]:
# Parte B + C — Pipeline con ColumnTransformer + LazyPredict 👇



In [ ]:
# Parte D + E — GridSearchCV + guardar el mejor modelo 👇



---
## 📌 Cierre del día y de la SEMANA

Esta semana cubrimos los pilares de la regresión profesional:

- ✅ **Lunes:** Pipelines y ColumnTransformer (automatización)
- ✅ **Martes:** SimpleImputer y joblib (producción)
- ✅ **Miércoles:** Random Forest y Feature Importance (modelos potentes)
- ✅ **Jueves:** Benchmarking + GridSearchCV + LazyPredict (decidir con evidencia)

### 🔜 Mañana — Viernes 29

**Simulacro de presentación de proyectos** — los equipos presentan lo que han trabajado.

### 🔜 Próxima semana — Semana 7: Clasificación

- Lun 1: Intro a clasificación + KNN Clasificador
- Mar 2: Métricas de clasificación (accuracy, precision, recall, F1, matriz de confusión)
- Mié 3: Regresión Logística
- Jue 4: Decision Tree y Random Forest Classifier
- Vie 5: Proyecto integrador: Calidad del vino

Nos vemos 🚀